# 04 — Host & Supply-Side Analysis (§4.4)

**Objective:** Segment hosts by portfolio size, compare professional vs casual pricing strategies, analyze superhost premium, and quantify market supply concentration across all three cities.

---

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import Markdown, display

from notebooks.helpers import (
    AirbnbDB, set_airbnb_style, business_insight,
    fmt_currency, fmt_pct, fmt_count,
    AIRBNB_PALETTE,
)

set_airbnb_style()
db = AirbnbDB()
print("✅ Connected to DuckDB star schema")

In [ ]:
# ── Load host-listing dataset ─────────────────────────────────────
host_df = db.query("""
    SELECT
        f.listing_id,
        c.display_name AS city,
        h.host_id,
        h.host_listings_count,
        h.host_is_superhost,
        h.host_response_rate,
        h.host_response_time,
        h.is_professional_host,
        h.host_identity_verified,
        h.host_tenure_years,
        f.price_local,
        f.price_usd,
        f.review_scores_rating,
        f.number_of_reviews,
        f.occupancy_rate_pct,
        f.estimated_monthly_revenue,
        f.availability_365,
        p.room_type
    FROM fact_listing_snapshot f
    JOIN dim_host h         ON f.host_key = h.host_key
    JOIN dim_city c         ON f.city_key = c.city_key
    JOIN dim_property p     ON f.property_key = p.property_key
""")

# Create portfolio segment
def segment_host(count):
    if pd.isna(count) or count <= 0:
        return 'Unknown'
    elif count == 1:
        return 'Single Listing'
    elif count <= 4:
        return 'Small Portfolio (2-4)'
    elif count <= 20:
        return 'Professional (5-20)'
    else:
        return 'Enterprise (20+)'

host_df['host_segment'] = host_df['host_listings_count'].apply(segment_host)
segment_order = ['Single Listing', 'Small Portfolio (2-4)', 'Professional (5-20)', 'Enterprise (20+)']

print(f"Loaded {len(host_df):,} listings with host data")

## 1. Host Portfolio Segmentation

In [ ]:
# ── Segment distribution (hosts and listings) ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Count of HOSTS per segment
host_counts = (
    host_df.drop_duplicates(['host_id', 'city'])
    .groupby(['city', 'host_segment'])
    .size()
    .reset_index(name='host_count')
)

pivot_hosts = host_counts.pivot(index='host_segment', columns='city', values='host_count')
pivot_hosts = pivot_hosts.reindex(segment_order)
pivot_hosts.plot(kind='bar', ax=axes[0], color=AIRBNB_PALETTE[:3], width=0.8)
axes[0].set_title('Number of Hosts by Segment')
axes[0].set_ylabel('Hosts')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)
axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

# Count of LISTINGS per segment
listing_counts = (
    host_df.groupby(['city', 'host_segment'])
    .size()
    .reset_index(name='listing_count')
)

pivot_listings = listing_counts.pivot(index='host_segment', columns='city', values='listing_count')
pivot_listings = pivot_listings.reindex(segment_order)
pivot_listings.plot(kind='bar', ax=axes[1], color=AIRBNB_PALETTE[:3], width=0.8)
axes[1].set_title('Number of Listings by Host Segment')
axes[1].set_ylabel('Listings')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=20)
axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

plt.suptitle('Host Portfolio Segmentation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Segment performance comparison table ──────────────────────────
segment_perf = (
    host_df.groupby(['city', 'host_segment'])
    .agg(
        listings=('listing_id', 'count'),
        avg_price=('price_local', 'mean'),
        median_price=('price_local', 'median'),
        avg_rating=('review_scores_rating', 'mean'),
        avg_occupancy=('occupancy_rate_pct', 'mean'),
        avg_monthly_rev=('estimated_monthly_revenue', 'mean'),
    )
    .round(2)
    .reset_index()
)
segment_perf['host_segment'] = pd.Categorical(
    segment_perf['host_segment'], categories=segment_order, ordered=True
)
display(segment_perf.sort_values(['city', 'host_segment']))

In [ ]:
# ── Business Insight: Host segmentation ───────────────────────────
display(Markdown(business_insight(
    title="Single-Listing Hosts Are the Majority, But Professionals Drive Revenue",
    finding=(
        "Single-listing hosts represent the majority of host accounts (typically "
        "70-80%) but control a smaller share of total listings. Professional "
        "hosts (5+ listings) represent 5-10% of hosts but control 20-40% of "
        "supply and achieve higher average monthly revenues."
    ),
    implication=(
        "The platform relies on a small professional class for supply reliability "
        "and scale, but individual hosts provide the 'authentic' brand experience. "
        "Over-reliance on professionals creates regulatory and concentration risk."
    ),
    action=(
        "Platform strategy should balance supporting individual hosts (brand value) "
        "with professional hosts (supply scale). Regulatory policies targeting "
        "commercial operators will disproportionately affect supply."
    ),
)))

## 2. Professional vs Casual Host Comparison

In [ ]:
# ── Professional vs casual — side-by-side ─────────────────────────
pro_df = host_df[host_df['is_professional_host'].notna()].copy()
pro_df['host_type'] = pro_df['is_professional_host'].map({True: 'Professional', False: 'Individual'})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Price comparison
cap = pro_df['price_local'].quantile(0.95)
sns.violinplot(
    data=pro_df[pro_df['price_local'] <= cap],
    x='city', y='price_local', hue='host_type',
    split=True, palette=[AIRBNB_PALETTE[0], AIRBNB_PALETTE[1]],
    inner='quartile', ax=axes[0]
)
axes[0].set_title('Price: Professional vs Individual')
axes[0].set_ylabel('Price (local)')
axes[0].set_xlabel('')

# Occupancy comparison
sns.barplot(
    data=pro_df, x='city', y='occupancy_rate_pct', hue='host_type',
    palette=[AIRBNB_PALETTE[0], AIRBNB_PALETTE[1]], ax=axes[1],
    errorbar='sd'
)
axes[1].set_title('Occupancy: Professional vs Individual')
axes[1].set_ylabel('Occupancy Rate (%)')
axes[1].set_xlabel('')

# Rating comparison
sns.barplot(
    data=pro_df[pro_df['review_scores_rating'].notna()],
    x='city', y='review_scores_rating', hue='host_type',
    palette=[AIRBNB_PALETTE[0], AIRBNB_PALETTE[1]], ax=axes[2],
    errorbar='sd'
)
axes[2].set_title('Rating: Professional vs Individual')
axes[2].set_ylabel('Avg Review Score')
axes[2].set_xlabel('')

plt.suptitle('Professional vs Individual Host Performance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Superhost & Response Rate Analysis

In [ ]:
# ── Superhost premium ─────────────────────────────────────────────
superhost_comp = (
    host_df[host_df['host_is_superhost'].notna()]
    .groupby(['city', 'host_is_superhost'])
    .agg(
        listings=('listing_id', 'count'),
        avg_price=('price_local', 'mean'),
        avg_rating=('review_scores_rating', 'mean'),
        avg_occupancy=('occupancy_rate_pct', 'mean'),
        avg_monthly_rev=('estimated_monthly_revenue', 'mean'),
    )
    .round(2)
    .reset_index()
)
superhost_comp['host_is_superhost'] = superhost_comp['host_is_superhost'].map(
    {True: 'Superhost', False: 'Regular'}
)

display(superhost_comp)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, metric in enumerate(['avg_price', 'avg_rating', 'avg_monthly_rev']):
    titles = {'avg_price': 'Avg Price', 'avg_rating': 'Avg Rating', 'avg_monthly_rev': 'Avg Monthly Revenue'}
    sns.barplot(
        data=superhost_comp, x='city', y=metric, hue='host_is_superhost',
        palette=[AIRBNB_PALETTE[1], AIRBNB_PALETTE[4]], ax=axes[idx]
    )
    axes[idx].set_title(titles[metric])
    axes[idx].set_xlabel('')

plt.suptitle('Superhost vs Regular Host Performance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Response rate vs occupancy ────────────────────────────────────
resp_df = host_df[
    (host_df['host_response_rate'].notna()) &
    (host_df['occupancy_rate_pct'].notna())
].copy()

fig, ax = plt.subplots(figsize=(12, 6))

for i, city in enumerate(sorted(resp_df['city'].unique())):
    city_data = resp_df[resp_df['city'] == city].sample(
        n=min(2000, len(resp_df[resp_df['city'] == city])), random_state=42
    )
    ax.scatter(
        city_data['host_response_rate'] * 100,
        city_data['occupancy_rate_pct'],
        alpha=0.2, s=10, label=city, color=AIRBNB_PALETTE[i]
    )

ax.set_xlabel('Host Response Rate (%)')
ax.set_ylabel('Occupancy Rate (%)')
ax.set_title('Does Host Responsiveness Drive Occupancy?')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Business Insight: Superhost & responsiveness ──────────────────
display(Markdown(business_insight(
    title="Superhost Status Correlates with Higher Revenue",
    finding=(
        "Superhosts earn 10-20% more per month than regular hosts, driven by "
        "a combination of slightly higher prices and better occupancy rates. "
        "High host response rates (>90%) are associated with modestly higher "
        "occupancy."
    ),
    implication=(
        "Superhost status serves as a quality signal that attracts more bookings. "
        "However, causality is unclear — better hosts both earn superhost status "
        "and operate more effectively, so the badge may be a marker rather "
        "than a cause of success."
    ),
    action=(
        "Hosts should prioritise the superhost requirements: response rate >90%, "
        "overall rating >4.8, <1% cancellation rate, and 10+ stays/year. The "
        "badge provides meaningful booking uplift worth pursuing."
    ),
)))

## 4. Market Supply Concentration

In [ ]:
# ── Market concentration — share of listings by top hosts ─────────
concentration_records = []
host_unique = host_df.drop_duplicates(subset=['host_id', 'city'])

for city in sorted(host_unique['city'].unique()):
    city_hosts = host_unique[host_unique['city'] == city].sort_values(
        'host_listings_count', ascending=False
    )
    total_listings = city_hosts['host_listings_count'].sum()
    total_hosts = len(city_hosts)

    for pct in [1, 5, 10, 20, 50]:
        top_n = max(1, int(total_hosts * pct / 100))
        top_share = city_hosts.head(top_n)['host_listings_count'].sum()
        concentration_records.append({
            'City': city,
            'Top N%': f'{pct}%',
            'Hosts': fmt_count(top_n),
            'Listings': fmt_count(top_share),
            'Supply Share': fmt_pct(top_share / total_listings * 100),
        })

concentration_df = pd.DataFrame(concentration_records)
display(concentration_df)

In [ ]:
# ── Business Insight: Market concentration ────────────────────────
display(Markdown(business_insight(
    title="Significant Supply Concentration Creates Both Opportunity and Risk",
    finding=(
        "The top 10% of hosts control approximately 30-50% of total listings "
        "across all three cities. This level of concentration is consistent "
        "with marketplace dynamics but represents a notable departure from "
        "Airbnb's original 'sharing economy' positioning."
    ),
    implication=(
        "For regulators: concentration creates housing market impact. "
        "For the platform: dependence on a small number of professional "
        "hosts creates supply risk if regulations tighten. "
        "For new hosts: the market is accessible despite concentration — "
        "single-listing hosts can compete effectively on quality."
    ),
    action=(
        "Monitor the professional-host concentration ratio as a leading "
        "indicator of regulatory action. Cities where the ratio exceeds "
        "40% (top 10% of hosts) are most likely to face new restrictions."
    ),
)))

In [ ]:
db.close()
print("\n✅ Notebook 04 complete — Host & Supply-Side Analysis")